In [1]:
!pip install -q zarr

import os
import math
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import concurrent.futures
from tqdm.auto import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 68.8 MB/s eta 0:00:00:00:0100:01


In [2]:
ruta_fuentes = "/kaggle/input/datasets/juanjoseorozcolopez/geovision-fuentes"
ruta_modis = "/kaggle/input/datasets/edwardsx/modis-v2-panel"
ruta_scl = "/kaggle/input/datasets/edwardsx/geovision-tiles-sit2/scl_por_escena.csv"

# Rutas de los paneles Zarr específicos
ruta_s2 = os.path.join(ruta_fuentes, "copernicus_s2_sr_harmonized", "panel.zarr")
ruta_s5p_no2 = os.path.join(ruta_fuentes, "copernicus_s5p_offl_l3_no2", "panel.zarr")
ruta_s5p_so2 = os.path.join(ruta_fuentes, "copernicus_s5p_offl_l3_so2", "panel.zarr")
ruta_s5p_o3 = os.path.join(ruta_fuentes, "copernicus_s5p_offl_l3_o3", "panel.zarr")
ruta_era5 = os.path.join(ruta_fuentes, "ecmwf_era5_hourly", "panel.zarr")
ruta_modis_zarr = os.path.join(ruta_modis, "panel.zarr")

# Ruta de salida para los nuevos conjuntos
ruta_salida = "/kaggle/working/geovision-tiles-v2-2021-2024"
os.makedirs(ruta_salida, exist_ok=True)

print(f"Rutas mapeadas. El output se guardará en: {ruta_salida}")

Rutas mapeadas. El output se guardará en: /kaggle/working/geovision-tiles-v2-2021-2024


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Apertura perezosa (lazy loading) con xarray
ds_s2 = xr.open_zarr(ruta_s2, chunks="auto")
ds_no2 = xr.open_zarr(ruta_s5p_no2, chunks="auto")
ds_so2 = xr.open_zarr(ruta_s5p_so2, chunks="auto")
ds_o3 = xr.open_zarr(ruta_s5p_o3, chunks="auto")
ds_era5 = xr.open_zarr(ruta_era5, chunks="auto")
ds_modis = xr.open_zarr(ruta_modis_zarr, chunks="auto")

# Carga de metadatos SCL de las escenas
df_scl = pd.read_csv(ruta_scl)

In [4]:
df_scl["fecha"] = pd.to_datetime(df_scl["time_s2"].str.split("T").str[0], format="%Y%m%d")
df_scl_validas = df_scl[
    (df_scl["fecha"] >= "2021-01-01") & 
    (df_scl["fecha"] <= "2024-12-31") & 
    (df_scl["scl_pct"] >= 0.30)
].copy()
df_scl_validas["fecha_s2_dt"] = pd.to_datetime(df_scl_validas["time_s2"].str.split("_").str[0], format="%Y%m%dT%H%M%S")

# Recorte temporal y enmascaramiento por nubosidad en Sentinel-5P (S5P)
ds_no2 = ds_no2.sel(time=slice("20210101", "20241231T235959"))
ds_so2 = ds_so2.sel(time=slice("20210101", "20241231T235959"))
ds_o3 = ds_o3.sel(time=slice("20210101", "20241231T235959"))

no2_cloud = ds_no2["data"].isel(band=2)
so2_cloud = ds_so2["data"].isel(band=1)
o3_cloud = ds_o3["data"].isel(band=1)
o3_val = ds_o3["data"].isel(band=0)

ds_no2["data"] = ds_no2["data"].where(no2_cloud < 0.7)
ds_so2["data"] = ds_so2["data"].where(so2_cloud < 0.7)
ds_o3["data"] = ds_o3["data"].where((o3_cloud < 0.7) & (o3_val > 0.0))

# Preprocesamiento de coordenadas meteorológicas ERA5 y MODIS
tiempos_era5 = pd.to_datetime(ds_era5["time"].values, format="%Y%m%dT%H")
ds_era5 = ds_era5.assign_coords(time=tiempos_era5).sel(time=slice("2021-01-01", "2024-12-31T23:59:59"))

ds_modis = ds_modis.sel(time=slice("A2021001", "A2024366"))
fechas_modis = pd.to_datetime([str(t)[1:] for t in ds_modis["time"].values], format="%Y%j")
ds_modis = ds_modis.assign_coords(time=fechas_modis)

In [5]:
def mapear_tiempos_s5p(ds, nombre):
    serie = ds["data"].isel(band=0).mean(dim=["x", "y"], skipna=True)
    df_temp = serie.to_dataframe(name=nombre).dropna().reset_index()
    df_temp["fecha"] = pd.to_datetime(df_temp["time"].astype(str).str.split("_").str[0], format="%Y%m%dT%H%M%S")
    return df_temp[["time", "fecha", nombre]]

df_t_no2 = mapear_tiempos_s5p(ds_no2, "no2")
df_t_so2 = mapear_tiempos_s5p(ds_so2, "so2")
df_t_o3 = mapear_tiempos_s5p(ds_o3, "o3")

In [6]:
def buscar_s5p_cercano(df_tiempos, fecha_objetivo, ventana_dias=1):
    fecha_objetivo = pd.Timestamp(fecha_objetivo)
    delta = (df_tiempos["fecha"] - fecha_objetivo).abs()
    mask = delta <= pd.Timedelta(days=ventana_dias)
    if not mask.any():
        return None
    return df_tiempos.loc[delta[mask].idxmin(), "time"]

def extraer_punto_s5p(ds, lat, lon, time_value, band_idx=0, ventana=1):
    ds_time = ds.sel(time=time_value)
    y_vals, x_vals = ds_time["y"].values, ds_time["x"].values
    iy = int(np.abs(y_vals - lat).argmin())
    ix = int(np.abs(x_vals - lon).argmin())
    y0, y1 = max(0, iy - ventana), min(len(y_vals), iy + ventana + 1)
    x0, x1 = max(0, ix - ventana), min(len(x_vals), ix + ventana + 1)
    bloque = ds_time["data"].isel(band=band_idx, y=slice(y0, y1), x=slice(x0, x1)).values
    return float(np.nanmean(bloque)) if np.isfinite(bloque).any() else np.nan

def extraer_s5p_cercano_en_punto(ds, df_tiempos, lat, lon, fecha_s2, ventana_dias=1, ventana_espacial=1):
    fecha_s2 = pd.Timestamp(fecha_s2)
    delta = (df_tiempos["fecha"] - fecha_s2).abs()
    candidatos = df_tiempos[delta <= pd.Timedelta(days=ventana_dias)].copy()
    if candidatos.empty:
        return np.nan
    candidatos["delta"] = (candidatos["fecha"] - fecha_s2).abs()
    candidatos = candidatos.sort_values("delta")
    for _, fila in candidatos.iterrows():
        valor = extraer_punto_s5p(ds, lat, lon, fila["time"], band_idx=0, ventana=ventana_espacial)
        if np.isfinite(valor):
            return valor
    return np.nan

def extraer_s5p_tile(lat, lon, fecha_s2):
    no2 = extraer_s5p_cercano_en_punto(ds_no2, df_t_no2, lat, lon, fecha_s2, ventana_dias=1, ventana_espacial=1)
    so2 = extraer_s5p_cercano_en_punto(ds_so2, df_t_so2, lat, lon, fecha_s2, ventana_dias=1, ventana_espacial=1)
    o3 = extraer_s5p_cercano_en_punto(ds_o3, df_t_o3, lat, lon, fecha_s2, ventana_dias=1, ventana_espacial=1)
    return no2, so2, o3

def extraer_modis_valor(lat, lon, fecha_s2, band_idx, ventana_espacial=2, ventana_dias=3):
    fecha_base = pd.Timestamp(fecha_s2).normalize()
    fechas_candidatas = [fecha_base + pd.Timedelta(days=d) for d in range(-ventana_dias, ventana_dias + 1)]
    fechas_candidatas = sorted(fechas_candidatas, key=lambda f: abs((f - fecha_base).days))
    for fecha in fechas_candidatas:
        if fecha < pd.Timestamp("2021-01-01") or fecha > pd.Timestamp("2024-12-31"):
            continue
        ds_time = ds_modis.sel(time=fecha)
        y_vals, x_vals = ds_time["y"].values, ds_time["x"].values
        iy = int(np.abs(y_vals - lat).argmin())
        ix = int(np.abs(x_vals - lon).argmin())
        y0, y1 = max(0, iy - ventana_espacial), min(len(y_vals), iy + ventana_espacial + 1)
        x0, x1 = max(0, ix - ventana_espacial), min(len(x_vals), ix + ventana_espacial + 1)
        bloque = ds_time["data"].isel(band=band_idx, y=slice(y0, y1), x=slice(x0, x1)).values
        if np.isfinite(bloque).any():
            return float(np.nanmean(bloque))
    return np.nan

def extraer_modis_tile(lat, lon, fecha_s2):
    return {
        "modis_AOD_047": extraer_modis_valor(lat, lon, fecha_s2, band_idx=0),
        "modis_AOD_055": extraer_modis_valor(lat, lon, fecha_s2, band_idx=1),
        "modis_WV": extraer_modis_valor(lat, lon, fecha_s2, band_idx=2),
    }

def extraer_era5_tile(lat, lon, fecha_s2):
    fecha_hora = pd.Timestamp(fecha_s2).round("h")
    punto = ds_era5.sel(time=fecha_hora, y=lat, x=lon, method="nearest")
    valores = punto["data"].values.astype("float32")
    return {
        "era5_T2m": float(valores[0]), "era5_Td2m": float(valores[1]),
        "era5_u10": float(valores[2]), "era5_v10": float(valores[3]),
        "era5_BLH": float(valores[4]), "era5_RH850": float(valores[5]),
        "era5_psurf": float(valores[6]), "era5_precip": float(valores[7]),
    }

In [7]:
TILE_SIZE = 64
SCL_TILE_MIN = 0.30
bands_s2 = list(ds_s2.coords["band"].values)

def extraer_tile_s2(escena, y0, x0, tile_size=64):
    return escena["data"].isel(y=slice(y0, y0 + tile_size), x=slice(x0, x0 + tile_size)).values.astype("float32")

def centroide_tile(escena, y0, x0, tile_size=64):
    y_c = y0 + tile_size // 2
    x_c = x0 + tile_size // 2
    return float(escena["y"].values[y_c]), float(escena["x"].values[x_c])

def calcular_indices_tile(tile, bands):
    idx_b4 = list(bands).index("B4")
    idx_b8 = list(bands).index("B8")
    idx_b11 = list(bands).index("B11")
    idx_scl = list(bands).index("SCL")
    
    red, nir, swir, scl = tile[idx_b4], tile[idx_b8], tile[idx_b11], tile[idx_scl]
    
    ndvi = np.nanmean((nir - red) / (nir + red + 1e-6))
    ndbi = np.nanmean((swir - nir) / (swir + nir + 1e-6))
    scl_pct = np.isin(scl, [4, 5, 6, 7]).mean()
    
    return float(ndvi), float(ndbi), float(scl_pct)

def muestrear_tile_valido(escena, bands, tile_size=64, scl_min=0.30, max_intentos=200):
    ny, nx = escena.sizes["y"], escena.sizes["x"]
    for _ in range(max_intentos):
        y0 = int(rng.integers(0, ny - tile_size))
        x0 = int(rng.integers(0, nx - tile_size))
        tile = extraer_tile_s2(escena, y0, x0, tile_size)
        ndvi, ndbi, scl_pct = calcular_indices_tile(tile, bands)
        if scl_pct >= scl_min and np.isfinite(ndvi) and np.isfinite(ndbi):
            lat, lon = centroide_tile(escena, y0, x0, tile_size)
            return {"tile": tile, "y0": y0, "x0": x0, "lat": lat, "lon": lon, "ndvi": ndvi, "ndbi": ndbi, "scl_pct": scl_pct}
    return None

In [8]:
import concurrent.futures
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from pathlib import Path
import datetime

# --- NUEVA FUNCIÓN DE CHECKPOINT ---
def guardar_checkpoint_paralelo(tiles, metas, intentos_evaluados):
    if len(tiles) == 0:
        return
    
    # Creamos una subcarpeta específica para no mezclar los checkpoints con el final
    ruta_ckpt = Path(ruta_salida) / "checkpoints_paralelos"
    ruta_ckpt.mkdir(parents=True, exist_ok=True)
    
    df_meta = pd.DataFrame(metas)
    tiles_arr = np.stack(tiles).astype("float32")
    
    # Guardamos versionando por la cantidad de intentos para trazabilidad histórica
    df_meta.to_parquet(ruta_ckpt / f"meta_ckpt_{intentos_evaluados}_intentos.parquet", index=False)
    np.savez_compressed(
        ruta_ckpt / f"tiles_ckpt_{intentos_evaluados}_intentos.npz",
        data=tiles_arr,
        bands=np.array(bands_s2)
    )
    
    # tqdm.write evita que el print rompa la barra de progreso visual en la consola
    hora = datetime.datetime.now().strftime("%H:%M:%S")
    tqdm.write(f"[{hora}] 💾 Checkpoint automático (Intento {intentos_evaluados}): Llevamos {len(tiles)}/{N_POR_CLASE} tiles correctos.")

# --- LÓGICA DE EXTRACCIÓN ---
def es_suelo_urbano_estricto(ndvi, ndbi):
    return ndbi > 0.05 and ndvi < 0.30

N_POR_CLASE = 230
OFFSET_SEED = 9999
WORKERS = 24
CHECKPOINT_FREQ = 300  # Frecuencia de guardado solicitada

def intentar_extraccion_urbana(intento):
    try:
        fila = df_scl_validas.sample(1, random_state=SEED + OFFSET_SEED + intento).iloc[0]
        time_s2, fecha_s2 = fila["time_s2"], fila["fecha_s2_dt"]
        escena = ds_s2.sel(time=time_s2)
        
        muestra = muestrear_tile_valido(escena, bands_s2, tile_size=TILE_SIZE, scl_min=SCL_TILE_MIN, max_intentos=50)
        if muestra is None:
            return None
            
        ndvi, ndbi = muestra["ndvi"], muestra["ndbi"]
        
        if es_suelo_urbano_estricto(ndvi, ndbi):
            lat, lon, scl_pct = muestra["lat"], muestra["lon"], muestra["scl_pct"]
            
            no2, so2, o3 = extraer_s5p_tile(lat, lon, fecha_s2)
            era5 = extraer_era5_tile(lat, lon, fecha_s2)
            modis = extraer_modis_tile(lat, lon, fecha_s2)
            
            texto = f"Zona urbana de alta densidad y suelo construido continuo, NDBI alto ({ndbi:.2f}) con firma vegetal ausente."
            
            meta = {
                "clase": "suelo_urbano_estricto",
                "time_s2": time_s2,
                "fecha_s2": fecha_s2,
                "lat": lat, "lon": lon,
                "ndvi": ndvi, "ndbi": ndbi, "scl_pct": scl_pct,
                "no2": no2, "so2": so2, "o3": o3,
                "texto": texto,
                **era5,
                **modis
            }
            return (muestra["tile"], meta)
    except Exception as e:
        pass
    return None

print(f"Iniciando extracción paralela con {WORKERS} workers...")
print(f"La trazabilidad se reportará cada {CHECKPOINT_FREQ} intentos evaluados.\n")

tiles_urbano = []
metas_urbano = []
intentos_evaluados = 0
MAX_INTENTOS = 12000

# Usamos ThreadPoolExecutor
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futuros = {executor.submit(intentar_extraccion_urbana, i): i for i in range(MAX_INTENTOS)}
    
    # as_completed entrega los futuros conforme van terminando (sin importar el orden en que empezaron)
    for futuro in tqdm(concurrent.futures.as_completed(futuros), total=MAX_INTENTOS, desc="Buscando tiles"):
        resultado = futuro.result()
        intentos_evaluados += 1
        
        if resultado is not None:
            tile, meta = resultado
            tiles_urbano.append(tile)
            metas_urbano.append(meta)
            
            # Condición de éxito
            if len(tiles_urbano) >= N_POR_CLASE:
                tqdm.write(f"\n✅ ¡Meta alcanzada! {N_POR_CLASE} muestras encontradas.")
                # Cancelamos los hilos restantes para ahorrar CPU
                for f in futuros:
                    f.cancel()
                break
        
        # --- TRAZABILIDAD Y CHECKPOINT ---
        # Se ejecuta exactamente cada 300 intentos evaluados
        if intentos_evaluados % CHECKPOINT_FREQ == 0:
            guardar_checkpoint_paralelo(tiles_urbano, metas_urbano, intentos_evaluados)

# --- GUARDADO FINAL ---
if len(tiles_urbano) > 0:
    ruta_export = Path(ruta_salida)
    ruta_export.mkdir(parents=True, exist_ok=True)
    
    # Aseguramos exactamente 230 muestras por si varios hilos terminaron en el mismo milisegundo
    tiles_urbano = tiles_urbano[:N_POR_CLASE]
    metas_urbano = metas_urbano[:N_POR_CLASE]
    
    df_meta_urbano = pd.DataFrame(metas_urbano)
    tiles_arr = np.stack(tiles_urbano).astype("float32")
    
    df_meta_urbano.to_parquet(ruta_export / "tiles_meta_urbano_estricto.parquet", index=False)
    np.savez_compressed(
        ruta_export / "tiles_train_urbano_estricto.npz",
        data=tiles_arr,
        bands=np.array(bands_s2)
    )
    print(f"\n🚀 Éxito definitivo. Archivos finales consolidados en: {ruta_export}")

Iniciando extracción paralela con 24 workers...
La trazabilidad se reportará cada 300 intentos evaluados.



Buscando tiles:   0%|          | 0/12000 [00:00<?, ?it/s]

[03:45:26] 💾 Checkpoint automático (Intento 300): Llevamos 12/230 tiles correctos.
[03:47:54] 💾 Checkpoint automático (Intento 600): Llevamos 37/230 tiles correctos.
[03:49:36] 💾 Checkpoint automático (Intento 900): Llevamos 58/230 tiles correctos.
[03:51:16] 💾 Checkpoint automático (Intento 1200): Llevamos 84/230 tiles correctos.
[03:53:15] 💾 Checkpoint automático (Intento 1500): Llevamos 99/230 tiles correctos.
[03:54:53] 💾 Checkpoint automático (Intento 1800): Llevamos 118/230 tiles correctos.
[03:57:02] 💾 Checkpoint automático (Intento 2100): Llevamos 133/230 tiles correctos.
[03:58:54] 💾 Checkpoint automático (Intento 2400): Llevamos 154/230 tiles correctos.
[04:00:41] 💾 Checkpoint automático (Intento 2700): Llevamos 175/230 tiles correctos.
[04:02:26] 💾 Checkpoint automático (Intento 3000): Llevamos 191/230 tiles correctos.
[04:04:06] 💾 Checkpoint automático (Intento 3300): Llevamos 212/230 tiles correctos.

✅ ¡Meta alcanzada! 230 muestras encontradas.

🚀 Éxito definitivo. Archiv

In [9]:
import os
import json
import subprocess
from pathlib import Path

# 1. Definición de rutas y variables
# Asegúrate de que ruta_salida sea la misma que definimos en las celdas anteriores
ruta_export = Path("/kaggle/working/geovision-tiles-v2-2021-2024")

# Reemplaza "tu_usuario" por tu handle de Kaggle (ej. "juanjoseorozcolopez" o "edwardsx")
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "edwardsx") 
DATASET_SLUG = "geovision-suelo-urbano-estricto"
DATASET_ID = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"

# 2. Creación del archivo de metadatos obligatorio para Kaggle
metadata = {
  "title": "GeoVision Cali - Suelo Urbano Estricto",
  "id": DATASET_ID,
  "licenses": [{"name": "CC0-1.0"}],
  "subtitle": "Clase urbana corregida para evitar colapso del espacio latente",
  "description": "Dataset corregido de la clase suelo_urbano (NDBI > 0.05 y NDVI < 0.30) para la Situación 2 del proyecto GeoVision-CLIP.",
  "keywords": ["remote sensing", "clip", "urban", "cali", "sentinel-2"]
}

with open(ruta_export / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata configurada para el dataset: {DATASET_ID}")

# 3. Función auxiliar para ejecutar la CLI de Kaggle e imprimir logs
def run_kaggle_cmd(cmd):
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout: 
        print("STDOUT:", proc.stdout.strip())
    if proc.stderr: 
        print("STDERR:", proc.stderr.strip())
    return proc

# 4. Despliegue en la nube de Kaggle
print("\n=== Iniciando subida a Kaggle Datasets ===")
comando_create = ["kaggle", "datasets", "create", "-p", str(ruta_export), "--dir-mode", "zip"]
proc_create = run_kaggle_cmd(comando_create)

# Lógica de contingencia: si el dataset ya existe, se sube como nueva versión
salida_combinada = (proc_create.stdout + proc_create.stderr).lower()
if proc_create.returncode != 0 and ("already exists" in salida_combinada or "409" in salida_combinada):
    print("\nEl dataset ya existe. Subiendo como una nueva versión (versioning)...")
    comando_version = ["kaggle", "datasets", "version", "-p", str(ruta_export), "-m", "Actualización de clase urbana estricta", "--dir-mode", "zip"]
    run_kaggle_cmd(comando_version)

print(f"\n✅ Proceso finalizado. Cuando termine de procesar, tu dataset estará en:\nhttps://www.kaggle.com/datasets/{DATASET_ID}")

Metadata configurada para el dataset: edwardsx/geovision-suelo-urbano-estricto

=== Iniciando subida a Kaggle Datasets ===
STDOUT: Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.0), please consider upgrading to the latest version (2.0.2)
Starting upload for file tiles_meta_urbano_estricto.parquet
Upload successful: tiles_meta_urbano_estricto.parquet (40KB)
Starting upload for file checkpoints_paralelos.zip
Upload successful: checkpoints_paralelos.zip (61MB)
Starting upload for file tiles_train_urbano_estricto.npz
Upload successful: tiles_train_urbano_estricto.npz (11MB)
The following are not valid tags and could not be added to the dataset: ['remote sensing', 'clip', 'urban', 'cali', 'sentinel-2']
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/edwardsx/geovision-suelo-urbano-estricto
STDERR: 0%|          | 0.00/40.4k [00:00<?, ?B/s]
100%|██████████| 40.4k/40.4k [00:00<00:00, 118kB/s]

  0%|          | 0.00/